# Rung 4 -- possession-outcome step heads (Colab training notebook)

CLAUDE.md ladder rung 4, scope per `NEXT_SESSION.md` item 4: five heads on one MLP trunk --
**outcome** (6-way), **zone** (3-way, shot attempts), **make** (binary, shot attempts),
**rebound** (binary, missed shots), **duration** (regression, seconds). No shooter head / DeepSets.

**Click path:** put `possession_steps.parquet` in Drive at `MyDrive/nba_colab/`, set Runtime > GPU,
Runtime > Run all. Artifacts land in `MyDrive/nba_colab/artifacts/<timestamp>/`.

**Inputs**: team-level as-of ratings + game state (10 cols) and, by default, as-of PLAYER/LINEUP features
(`off_pa_*`/`def_pa_*`: mean/max/min pooling over the 5 on-court players of strictly-prior shrunk player
stats). Set env `NBA_FEATURE_SET=team` for the team-only ablation. Re-export the parquet after pulling this version.

**Self-contained**: everything below is duplicated from (and kept in sync, by a test, with)
`nba/features/possession_step_features.py` and `nba/models/rung4_stepheads.py`.

**Leakage / holdout discipline** (`docs/ACCEPTANCE_CRITERIA_2026-10-08.md` section 3): features are
strictly-prior as-of team ratings or intra-game state known before the possession's outcome. Tuning and
early stopping use `season < HOLDOUT_SEASON` only (chronological split by `game_date`). The frozen holdout
(`HOLDOUT_SEASON = 2025`, the 2025-26 season) is scored ONCE at the end, using the best-validation weights.
That single confirmatory evaluation must be pre-registered in `docs/HOLDOUT_ACCESS_LOG.md` before you run.
Do not re-run with changed hyperparameters after seeing the holdout numbers.


In [ ]:
# --- Config + environment (Drive-aware) ---------------------------------
import json
import os
import random
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import polars as pl
import torch
from torch import nn

try:  # Colab?
    from google.colab import drive  # type: ignore

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/nba_colab")
    PARQUET_PATH = DRIVE_DIR / "possession_steps.parquet"
    ARTIFACT_ROOT = DRIVE_DIR / "artifacts"
else:  # local fallback (run from the repo root)
    PARQUET_PATH = Path("data/colab/possession_steps.parquet")
    ARTIFACT_ROOT = Path("data/colab/artifacts")

SEED = 0
HIDDEN = 128
LR = 1e-3
N_EPOCHS = 60
BATCH_SIZE = 4096
PATIENCE = 5  # early stop on within-tunable validation loss (never the holdout)
# Frozen final holdout = the 2025-26 season, explicit (NOT df.season.max()).
HOLDOUT_SEASON = 2025
SCORE_HOLDOUT = True  # the ONE confirmatory evaluation; pre-register it first
SAMPLE_ROWS = None  # int -> strided subsample for a quick smoke run

# Optional env overrides (used by local dry runs; harmless in Colab).
PARQUET_PATH = Path(os.environ.get("NBA_PARQUET", PARQUET_PATH))
ARTIFACT_ROOT = Path(os.environ.get("NBA_ARTIFACT_ROOT", ARTIFACT_ROOT))
N_EPOCHS = int(os.environ.get("NBA_N_EPOCHS", N_EPOCHS))
HIDDEN = int(os.environ.get("NBA_HIDDEN", HIDDEN))
HOLDOUT_SEASON = int(os.environ.get("NBA_HOLDOUT_SEASON", HOLDOUT_SEASON))
SCORE_HOLDOUT = os.environ.get("NBA_SCORE_HOLDOUT", str(SCORE_HOLDOUT)) == "True"
if os.environ.get("NBA_SAMPLE_ROWS"):
    SAMPLE_ROWS = int(os.environ["NBA_SAMPLE_ROWS"])

OUT_DIR = ARTIFACT_ROOT / datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR.mkdir(parents=True, exist_ok=True)
assert PARQUET_PATH.exists(), (
    f"parquet not found: {PARQUET_PATH} (README: put it in Drive nba_colab/)"
)

# --- Device: GPU check ----------------------------------------------------
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print("device: cuda ->", torch.cuda.get_device_name(0))
else:
    print("!" * 70)
    print("WARNING: NO GPU. Runtime > Change runtime type > T4 GPU, then Run all again.")
    print("The model is tiny so CPU still works, but it will be slow on ~1M rows.")
    print("!" * 70)


# --- Seed everything + log it ---------------------------------------------
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
print(f"SEED={SEED} | HOLDOUT_SEASON={HOLDOUT_SEASON} | parquet={PARQUET_PATH} | out={OUT_DIR}")

In [ ]:
import os

# --- Constants (keep in sync with nba/features/possession_step_features.py) ---
OUTCOME_CLASSES = ["FGM2", "FGM3", "FGA_miss", "TOV", "FT_trip", "other"]
ZONE_CLASSES = ["rim", "mid", "above3"]
SHOT_ATTEMPT_OUTCOMES = ["FGM2", "FGM3", "FGA_miss"]
MADE_OUTCOMES = ["FGM2", "FGM3"]
POSSESSION_STEP_TEAM_FEATURE_COLUMNS = [
    "period",
    "clock_start",
    "score_diff",
    "off_is_home",
    "off_off_rtg_prior",
    "off_def_rtg_prior",
    "off_pace_prior",
    "def_off_rtg_prior",
    "def_def_rtg_prior",
    "league_avg_ppp_asof",
]
PLAYER_ASOF_STATS = [
    "usage", "zmix_rim", "zmix_mid", "zmix_above3", "zfg_rim", "zfg_mid", "zfg_above3",
    "tov", "oreb", "oreb_allowed", "forced_tov", "opp_make", "logn",
]  # fmt: skip
PLAYER_POOL_AGGS = ["mean", "max", "min"]
POSSESSION_STEP_PLAYER_FEATURE_COLUMNS = [
    f"{side}_pa_{s}_{a}"
    for side in ("off", "def")
    for s in PLAYER_ASOF_STATS
    for a in PLAYER_POOL_AGGS
]
POSSESSION_STEP_FEATURE_COLUMNS = (
    POSSESSION_STEP_TEAM_FEATURE_COLUMNS + POSSESSION_STEP_PLAYER_FEATURE_COLUMNS
)
# Model input columns: "team+player" (default) or the team-only ablation.
FEATURE_COLUMNS = (
    POSSESSION_STEP_TEAM_FEATURE_COLUMNS
    if os.environ.get("NBA_FEATURE_SET", "team+player") == "team"
    else POSSESSION_STEP_FEATURE_COLUMNS
)
POSSESSION_STEP_LABEL_COLUMNS = ["outcome", "shot_zone", "made_shot", "oreb", "duration_s"]
DURATION_MAX_S = 60.0
DURATION_LOSS_WEIGHT = 0.001


def derive_step_labels(raw):
    """Raw parquet -> training frame (copy of possession_step_features.derive_step_labels)."""
    raw = raw.filter(pl.col("outcome").is_in(OUTCOME_CLASSES))
    is_shot = pl.col("outcome").is_in(SHOT_ATTEMPT_OUTCOMES)
    made = pl.col("outcome").is_in(MADE_OUTCOMES)
    duration = (pl.col("clock_start") - pl.col("clock_end")).clip(0.0, DURATION_MAX_S)
    out = raw.with_columns(
        [
            pl.col("off_is_home").cast(pl.Int64),
            made.cast(pl.Int64).alias("made_shot"),
            pl.when(is_shot).then(pl.col("shot_zone")).otherwise(None).alias("shot_zone"),
            pl.col("oreb").fill_null(False),
            duration.alias("duration_s"),
        ]
    ).filter(~is_shot | pl.col("shot_zone").is_in(ZONE_CLASSES))
    keep = [
        "game_id",
        "poss_idx",
        "game_date",
        "season",
        *POSSESSION_STEP_FEATURE_COLUMNS,
        *POSSESSION_STEP_LABEL_COLUMNS,
    ]
    return out.select(keep)


# --- Model (keep in sync with nba/models/rung4_stepheads.py) ------------
class StepHeadsNet(nn.Module):
    def __init__(self, n_features, hidden=64):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )
        self.outcome_head = nn.Linear(hidden, len(OUTCOME_CLASSES))
        self.zone_head = nn.Linear(hidden, len(ZONE_CLASSES))
        self.make_head = nn.Linear(hidden, 1)
        self.rebound_head = nn.Linear(hidden, 1)
        self.duration_head = nn.Linear(hidden, 1)

    def forward(self, x):
        h = self.trunk(x)
        return {
            "outcome_logits": self.outcome_head(h),
            "zone_logits": self.zone_head(h),
            "make_logit": self.make_head(h).squeeze(-1),
            "rebound_logit": self.rebound_head(h).squeeze(-1),
            "duration": self.duration_head(h).squeeze(-1),
        }


def feature_matrix(df, mean=None, std=None):
    raw = df.select(FEATURE_COLUMNS).to_numpy().astype(np.float64)
    mean = raw.mean(axis=0) if mean is None else mean
    std = raw.std(axis=0) if std is None else std
    std = np.where(std < 1e-8, 1.0, std)
    return (raw - mean) / std, mean, std


def frame_to_tensors(df, mean=None, std=None):
    x, mean, std = feature_matrix(df, mean, std)
    o_idx = {c: i for i, c in enumerate(OUTCOME_CLASSES)}
    z_idx = {c: i for i, c in enumerate(ZONE_CLASSES)}
    outcomes = df.get_column("outcome").to_list()
    zones = df.get_column("shot_zone").to_list()
    tensors = {
        "x": torch.tensor(x, dtype=torch.float32),
        "outcome_idx": torch.tensor([o_idx[o] for o in outcomes], dtype=torch.long),
        "is_shot": torch.tensor([o in SHOT_ATTEMPT_OUTCOMES for o in outcomes], dtype=torch.bool),
        "zone_idx": torch.tensor([z_idx.get(z, 0) for z in zones], dtype=torch.long),
        "made": torch.tensor(df.get_column("made_shot").to_numpy().astype(np.float32)),
        "is_miss": torch.tensor([o == "FGA_miss" for o in outcomes], dtype=torch.bool),
        "oreb": torch.tensor(df.get_column("oreb").to_numpy().astype(np.float32)),
        "duration": torch.tensor(df.get_column("duration_s").to_numpy().astype(np.float32)),
    }
    return tensors, mean, std


def compute_losses(net, batch):
    out = net(batch["x"])
    zero = torch.tensor(0.0, device=batch["x"].device)
    outcome_loss = nn.functional.cross_entropy(out["outcome_logits"], batch["outcome_idx"])
    is_shot = batch["is_shot"]
    if is_shot.any():
        zone_loss = nn.functional.cross_entropy(
            out["zone_logits"][is_shot], batch["zone_idx"][is_shot]
        )
        make_loss = nn.functional.binary_cross_entropy_with_logits(
            out["make_logit"][is_shot], batch["made"][is_shot]
        )
    else:
        zone_loss, make_loss = zero, zero
    is_miss = batch["is_miss"]
    if is_miss.any():
        rebound_loss = nn.functional.binary_cross_entropy_with_logits(
            out["rebound_logit"][is_miss], batch["oreb"][is_miss]
        )
    else:
        rebound_loss = zero
    duration_loss = nn.functional.mse_loss(out["duration"], batch["duration"])
    total = (
        outcome_loss + zone_loss + make_loss + rebound_loss + DURATION_LOSS_WEIGHT * duration_loss
    )
    return {
        "outcome": outcome_loss,
        "zone": zone_loss,
        "make": make_loss,
        "rebound": rebound_loss,
        "duration": duration_loss,
        "total": total,
    }


# --- Frequency baseline + per-head eval (copies of rung4_stepheads helpers) ---
def frequency_priors(b):
    oc = torch.bincount(b["outcome_idx"], minlength=len(OUTCOME_CLASSES)).double().clamp_min(0.5)
    zc = (
        torch.bincount(b["zone_idx"][b["is_shot"]], minlength=len(ZONE_CLASSES))
        .double()
        .clamp_min(0.5)
    )
    make = float(b["made"][b["is_shot"]].double().mean()) if b["is_shot"].any() else 0.5
    reb = float(b["oreb"][b["is_miss"]].double().mean()) if b["is_miss"].any() else 0.5
    return {
        "outcome": oc / oc.sum(),
        "zone": zc / zc.sum(),
        "make": min(max(make, 1e-6), 1 - 1e-6),
        "rebound": min(max(reb, 1e-6), 1 - 1e-6),
        "duration": float(b["duration"].double().mean()),
    }


def _bernoulli_ll(y, p):
    y = y.double()
    return float(-(y * np.log(p) + (1 - y) * np.log(1 - p)).mean())


def evaluate_heads(net, b, priors):
    net.eval()
    with torch.no_grad():
        out = net(b["x"])
        ce = nn.functional.cross_entropy
        bce = nn.functional.binary_cross_entropy_with_logits
        op, zp = priors["outcome"].to(b["x"].device), priors["zone"].to(b["x"].device)
        res = {
            "outcome": (
                float(ce(out["outcome_logits"], b["outcome_idx"])),
                float(-op.log()[b["outcome_idx"]].mean()),
            )
        }
        sh, ms = b["is_shot"], b["is_miss"]
        if sh.any():
            res["zone"] = (
                float(ce(out["zone_logits"][sh], b["zone_idx"][sh])),
                float(-zp.log()[b["zone_idx"][sh]].mean()),
            )
            res["make"] = (
                float(bce(out["make_logit"][sh], b["made"][sh])),
                _bernoulli_ll(b["made"][sh], priors["make"]),
            )
        if ms.any():
            res["rebound"] = (
                float(bce(out["rebound_logit"][ms], b["oreb"][ms])),
                _bernoulli_ll(b["oreb"][ms], priors["rebound"]),
            )
        res["duration"] = (
            float(nn.functional.mse_loss(out["duration"], b["duration"])),
            float(((b["duration"] - priors["duration"]) ** 2).mean()),
        )
    return {
        h: {
            "net": n,
            "baseline": base,
            "skill": 1.0 - n / base if base > 0 else float("nan"),
            "delta": n - base,
        }
        for h, (n, base) in res.items()
    }

In [ ]:
# --- Load parquet, derive labels, chronological split ----------------------
t0 = time.monotonic()
raw = pl.read_parquet(PARQUET_PATH)
print("raw rows:", raw.height, "| seasons:", sorted(raw["season"].unique().to_list()))
if SAMPLE_ROWS is not None and raw.height > SAMPLE_ROWS:
    raw = raw.gather_every(raw.height // SAMPLE_ROWS)  # strided: keeps all seasons, order intact
    print("SAMPLE_ROWS ->", raw.height, "rows (strided)")
df = derive_step_labels(raw)
print(f"after label derivation: {df.height} rows (dropped {raw.height - df.height})")
print("date range:", df["game_date"].min(), "to", df["game_date"].max())

# Fail fast on schema / NaN problems instead of mid-training.
need = POSSESSION_STEP_FEATURE_COLUMNS + ["outcome", "made_shot", "oreb", "duration_s"]
bad = {c: int(df[c].is_null().sum()) for c in need if df[c].is_null().sum() > 0}
assert not bad, f"nulls in required columns: {bad}"
feat_np = df.select(POSSESSION_STEP_FEATURE_COLUMNS).to_numpy().astype(np.float64)
assert np.isfinite(feat_np).all(), "non-finite feature values"
unseen = set(df["outcome"].unique().to_list()) - set(OUTCOME_CLASSES)
assert not unseen, f"unknown outcome labels: {unseen}"
print("outcome counts:", dict(df["outcome"].value_counts().iter_rows()))

# Frozen holdout is EXPLICIT (HOLDOUT_SEASON), tuning uses season < HOLDOUT_SEASON only.
if df["season"].max() != HOLDOUT_SEASON:
    print(f"NOTE: df.season.max()={df['season'].max()} != HOLDOUT_SEASON={HOLDOUT_SEASON}")
tunable = df.filter(pl.col("season") < HOLDOUT_SEASON)
holdout = df.filter(pl.col("season") == HOLDOUT_SEASON)
assert tunable.height > 0, "no tunable rows (season < HOLDOUT_SEASON)"
if SCORE_HOLDOUT:
    assert holdout.height > 0, f"no rows for HOLDOUT_SEASON={HOLDOUT_SEASON}"
print(
    f"tunable (season<{HOLDOUT_SEASON}): {tunable.height} rows | "
    f"frozen holdout (season=={HOLDOUT_SEASON}): {holdout.height} rows"
)

# Walk-forward inside the tunable set: last 15% of dates = validation (early stopping only).
dates = tunable.select("game_date").unique().sort("game_date").to_series()
cutoff = dates[int(len(dates) * 0.85)]
train_df = tunable.filter(pl.col("game_date") < cutoff)
val_df = tunable.filter(pl.col("game_date") >= cutoff)
assert train_df.height > 0 and val_df.height > 0, "empty train or val window"
print(f"train: {train_df.height} rows (< {cutoff}) | val: {val_df.height} rows (>= {cutoff})")

In [ ]:
# --- Train (early stopping on the within-tunable validation window) ---------
seed_everything(SEED)
train_batch, feat_mean, feat_std = frame_to_tensors(train_df)
val_batch, _, _ = frame_to_tensors(val_df, feat_mean, feat_std)
train_batch = {k: v.to(DEVICE) for k, v in train_batch.items()}
val_batch = {k: v.to(DEVICE) for k, v in val_batch.items()}
priors = frequency_priors(train_batch)  # baseline = TRAIN-window class priors only

net = StepHeadsNet(n_features=len(FEATURE_COLUMNS), hidden=HIDDEN).to(DEVICE)
optimizer = torch.optim.Adam(net.parameters(), lr=LR)

n = train_batch["x"].shape[0]
history, best_val, best_state, bad_epochs = [], float("inf"), None, 0
start = time.monotonic()
for epoch in range(N_EPOCHS):
    net.train()
    perm = torch.randperm(n, device=DEVICE)
    epoch_loss = 0.0
    for i in range(0, n, BATCH_SIZE):
        idx = perm[i : i + BATCH_SIZE]
        losses = compute_losses(net, {k: v[idx] for k, v in train_batch.items()})
        optimizer.zero_grad()
        losses["total"].backward()
        optimizer.step()
        epoch_loss += float(losses["total"].detach()) * idx.shape[0]
    epoch_loss /= n
    net.eval()
    with torch.no_grad():
        val_total = float(compute_losses(net, val_batch)["total"])
    assert np.isfinite(epoch_loss) and np.isfinite(val_total), "NaN/inf loss"
    history.append({"epoch": epoch, "train_loss": epoch_loss, "val_loss": val_total})
    print(f"epoch {epoch:3d} | train {epoch_loss:.4f} | val {val_total:.4f}")
    if val_total < best_val - 1e-5:
        best_val, bad_epochs = val_total, 0
        best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"early stop at epoch {epoch} (best val {best_val:.4f})")
            break
net.load_state_dict(best_state)  # restore best-validation weights
print(f"training took {time.monotonic() - start:.1f}s")

In [ ]:
# --- Per-head evaluation vs frequency baseline -----------------------------
val_heads = evaluate_heads(net, val_batch, priors)
print("VALIDATION (tunable window, tuning allowed):")
for h, r in val_heads.items():
    print(
        f"  {h:9s} net {r['net']:.4f} | freq {r['baseline']:.4f} | "
        f"skill {r['skill']:+.3%} | delta {r['delta']:+.5f}"
    )

holdout_heads, holdout_total = {}, {}
if SCORE_HOLDOUT:
    # THE one confirmatory evaluation (pre-register in docs/HOLDOUT_ACCESS_LOG.md).
    hb, _, _ = frame_to_tensors(holdout, feat_mean, feat_std)
    hb = {k: v.to(DEVICE) for k, v in hb.items()}
    holdout_heads = evaluate_heads(net, hb, priors)
    net.eval()
    with torch.no_grad():
        holdout_total = {k: float(v) for k, v in compute_losses(net, hb).items()}
    print(f"FROZEN HOLDOUT (season {HOLDOUT_SEASON}, n={holdout.height}):")
    for h, r in holdout_heads.items():
        print(f"  {h:9s} net {r['net']:.4f} | freq {r['baseline']:.4f} | skill {r['skill']:+.3%}")
    learned = [h for h in ("outcome", "zone", "make", "rebound") if holdout_heads[h]["skill"] > 0]
    print(
        "heads beating the frequency baseline on holdout:", learned or "NONE (net learned nothing)"
    )
else:
    print("SCORE_HOLDOUT=False -> holdout not touched.")

In [ ]:
# --- Save artifacts (Drive: no manual download needed) ----------------------
weights_path = OUT_DIR / "weights.pt"
torch.save({k: v.cpu() for k, v in net.state_dict().items()}, weights_path)

config = {
    "model_name": "rung4_stepheads",
    "rung": 4,
    "seed": SEED,
    "hidden": HIDDEN,
    "lr": LR,
    "n_epochs": N_EPOCHS,
    "epochs_run": len(history),
    "batch_size": BATCH_SIZE,
    "patience": PATIENCE,
    "feature_columns": FEATURE_COLUMNS,
    "feature_mean": feat_mean.tolist(),
    "feature_std": feat_std.tolist(),
    "outcome_classes": OUTCOME_CLASSES,
    "zone_classes": ZONE_CLASSES,
    "holdout_season": HOLDOUT_SEASON,
    "score_holdout": SCORE_HOLDOUT,
    "device": str(DEVICE),
    "sample_rows": SAMPLE_ROWS,
    "parquet": str(PARQUET_PATH),
    "n_train_rows": train_df.height,
    "n_val_rows": val_df.height,
    "n_holdout_rows": holdout.height,
    "train_val_cutoff_date": str(cutoff),
    "created": datetime.now().isoformat(timespec="seconds"),
}
metrics = {
    "history": history,
    "best_val_loss": best_val,
    "val_heads": val_heads,
    "holdout": holdout_total,
    "holdout_heads": holdout_heads,
    "baseline": "class priors / base rates / mean duration from the TRAIN window",
}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=2))
(OUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2))
print(f"weights: {weights_path} ({weights_path.stat().st_size / 1e6:.3f} MB)")
print(f"ARTIFACTS WRITTEN TO: {OUT_DIR}")